# Project 07: Burrows-Wheeler Transform

In [38]:
def suffix_array(string: str) -> list[int]:
    """Function to calculate suffix-array for a given string.
    
    Computes the suffix array by sorting all suffixes of the input string
    lexicographically and returning their starting positions.
    
    Args:
        string: The input string to process.
    
    Returns:
        A list of integers representing the starting positions of the
        lexicographically sorted suffixes.
        
    Examples:
        >>> suffix_array('googol')
        [6, 3, 0, 5, 2, 4, 1]
        
        >>> suffix_array('banana$')
        [6, 5, 3, 1, 0, 4, 2]
    """
    new_string = f'{string.lower()}$'
    string_indices = list(range(len(new_string)))
    suffix_list = []
    for index in string_indices: 
        suffix_list.append((index,new_string[index:]))
    sorted_suffix_list = sorted(suffix_list, key = lambda x:x[1])
    print(sorted_suffix_list)
    BWT = []
    for j in range(len(sorted_suffix_list)): 
        q = sorted_suffix_list[j][0] - 1
        BWT.append(new_string[q])
    print(BWT)
    return sorted_suffix_list, BWT

In [39]:
suf_list, BWT = suffix_array("Banana")

[(6, '$'), (5, 'a$'), (3, 'ana$'), (1, 'anana$'), (0, 'banana$'), (4, 'na$'), (2, 'nana$')]
['a', 'n', 'n', 'b', '$', 'a', 'a']


In [94]:
from pprint import pprint

def find_match(query: str, reference: str, right_col, left_col) -> list[int]:
    """Function to find exact matching by applying Burrows-Wheeler Transform.
    
    Searches for all occurrences of the query string within the reference string
    using the Burrows-Wheeler Transform algorithm for efficient pattern matching.
    
    Args:
        query: The pattern string to search for.
        reference: The text string to search within.
    
    Returns:
        A list of integers representing the 0-based starting positions of all
        occurrences of the query string within the reference string. Returns an
        empty list if no matches are found.
    
    Examples:
        >>> find_match('ana', 'banana')
        [1, 3]
        
        >>> find_match('xyz', 'banana')
        []
    """
    reference = reference.lower()
    query = query.lower()
    right_col = [char for char in right_col]       #casting as a list for simplicity
    left_col_ex = [entry[1][0] for entry in left_col] #extracting values from tuple 

    left_col_keys = set(right_col)  #just getting keys
    right_col_keys = set(left_col_ex)

    index_dict = {key: 0 for key in left_col_keys} #setting dict for initial indices

    found = set()                       #keeping track of what has been found so no overwriting
    for i in range(len(left_col_ex)):
        if left_col_ex[i] not in found:
            key = left_col_ex[i]
            index_dict[key] = i
            found.add(key)
    
    total_occurrences = {key: 0 for key in found}
    occurrence_arrays = {key: [0 for _ in range(len(right_col))] for key in right_col_keys}

    for i in range(len(right_col)): #iterate through characters
        char = right_col[i]
        total_occurrences[char] += 1
        for char in total_occurrences:
            occurrence_arrays[char][i] += total_occurrences[char]

    suffix_array = [entry[0] for entry in left_col]

    upper = len(suffix_array) - 1
    lower = 0
    for char in reversed(query):
        idx = index_dict[char]
        if lower <= 0:
            lower = idx
        else:
            lower = idx + occurrence_arrays[char][lower - 1]
        upper = idx + occurrence_arrays[char][upper] - 1
    
    idx = suffix_array[lower: upper + 1]
    output = {k: v for k, v in left_col}
    print([output[i] for i in idx])
    return idx
    


zz, zz_bwt = suffix_array("ATTGACCABBDSCCAGACADF")
find_match(query="BBD", reference="ATTGACCABBDSCCAGACADF", right_col=zz_bwt, left_col=zz)

[(21, '$'), (7, 'abbdsccagacadf$'), (16, 'acadf$'), (4, 'accabbdsccagacadf$'), (18, 'adf$'), (14, 'agacadf$'), (0, 'attgaccabbdsccagacadf$'), (8, 'bbdsccagacadf$'), (9, 'bdsccagacadf$'), (6, 'cabbdsccagacadf$'), (17, 'cadf$'), (13, 'cagacadf$'), (5, 'ccabbdsccagacadf$'), (12, 'ccagacadf$'), (19, 'df$'), (10, 'dsccagacadf$'), (20, 'f$'), (15, 'gacadf$'), (3, 'gaccabbdsccagacadf$'), (11, 'sccagacadf$'), (2, 'tgaccabbdsccagacadf$'), (1, 'ttgaccabbdsccagacadf$')]
['f', 'c', 'g', 'g', 'c', 'c', '$', 'a', 'b', 'c', 'a', 'c', 'a', 's', 'a', 'b', 'd', 'a', 't', 'd', 't', 'a']
['bbdsccagacadf$']


[8]

In [45]:
def run_length_encode(bwt_string: str) -> str:
    """Run-length encode a BWT string as a compact character-count string.

    Scans the input BWT string from left to right and groups consecutive
    identical characters into runs, returning an encoded string where each
    run is represented as the character followed by its count.

    Args:
    bwt_string: The Burrows-Wheeler transformed string to encode.

    Returns:
    A run-length encoded string of the form 'a3n2b1$1a2', where each
    character is followed by the length of its consecutive run.

    Examples:
    >>> run_length_encode('aaabbc')
    'a3b2c1'

    >>> run_length_encode('annb$aa')
    'a1n2b1$1a2'
    """
    encoded_bwt = []
    letter_counter = 1
    for i in range(len(bwt_string) - 1):
        #if i < len(bwt_string -1):
        if bwt_string[i] == bwt_string[i+1]:
            letter_counter += 1
            if i == len(bwt_string) - 2:
                encoded_bwt.append(f'{bwt_string[i]}{letter_counter}')      
        else:
            encoded_bwt.append(f'{bwt_string[i]}{letter_counter}')
            letter_counter = 1
    if letter_counter == 1:
        encoded_bwt.append(f'{bwt_string[-1]}{letter_counter}')

    return ''.join(encoded_bwt)   
        
        # else: 
        #     if bwt_string[-1] == bwt_string[-2]:
        #         letter_counter += 1
        #         encoded_bwt.append(f'{letter_counter}{bwt_string[i]}')
encoded_bwt = run_length_encode('annb$aa') 
print(encoded_bwt)                       


a1n2b1$1a2


In [46]:
def run_length_decode(encoded: str) -> str:
    """Decode a run-length encoded character-count string.

    Reconstructs the original string by parsing an encoded representation
    where each run is stored as a character followed by its count, and
    expanding each run back into repeated characters.

    Args:
    encoded: A run-length encoded string of the form 'a3n2b1$1a2'.

    Returns:
    The decoded string obtained by expanding all runs in the encoded input.

    Examples:
    >>> run_length_decode('a3b2c1')
    'aaabbc'

    >>> run_length_decode('a1n2b1$1a2')
    'annb$aa'
    """
    decoded_li = []
    for i in range(len(encoded) -1, -1, -1):
        if encoded[i].isnumeric() == False:
            decoded_li.append(encoded[i] * int(encoded[i + 1:]))
            encoded = encoded[:i]

    return ''.join(reversed(decoded_li))

run_length_decode(encoded_bwt)

'annb$aa'